# Day 1 — Why Polars exists: where pandas hurts at scale

*100 Days of Polars · Phase 1: Foundations*

**Goal for today:** feel the difference before learning the API. We run the same work in pandas and Polars on the Olist e-commerce data and measure load time, memory, and query time.

**What you'll leave with**
- A measured (not marketing) comparison on real, multi-table data
- The 4 design choices that make Polars fast
- Notes and numbers for the Day 1 post

In [ ]:
import time
from pathlib import Path

import pandas as pd
import polars as pl

# Works whether the notebook runs from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "raw"

print("polars", pl.__version__, "| pandas", pd.__version__)
print("Polars threads:", pl.thread_pool_size())
sorted(p.name for p in DATA.glob("*.csv"))

In [ ]:
class timer:
    """Tiny context manager: `with timer("label"): ...` prints elapsed seconds."""
    def __init__(self, label): self.label = label
    def __enter__(self): self.t0 = time.perf_counter(); return self
    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self.t0
        print(f"{self.label:<28} {self.seconds:6.2f} s")

## Experiment 1 — Load all 9 CSVs

Same files, default settings in both libraries. About 1.6 M rows in total, most of them in the geolocation table.

Run each cell **twice** and use the second timing: the first run includes disk caching.

In [ ]:
files = sorted(DATA.glob("*.csv"))

with timer("pandas: read 9 CSVs") as t_pd:
    pd_frames = {f.stem: pd.read_csv(f) for f in files}

with timer("polars: read 9 CSVs") as t_pl:
    pl_frames = {f.stem: pl.read_csv(f) for f in files}

print(f"speed-up: {t_pd.seconds / t_pl.seconds:.1f}x")

In [ ]:
# Memory: pandas needs deep=True to count Python string objects honestly
mem = pl.DataFrame({
    "table": list(pd_frames),
    "rows": [len(df) for df in pd_frames.values()],
    "pandas_mb": [df.memory_usage(deep=True).sum() / 1e6 for df in pd_frames.values()],
    "polars_mb": [pl_frames[k].estimated_size("mb") for k in pd_frames],
}).with_columns(ratio=pl.col("pandas_mb") / pl.col("polars_mb")).sort("pandas_mb", descending=True)

mem

## Experiment 2 — A realistic query

**Monthly revenue per product category (English names) for delivered orders.**

Four tables: `orders` → `order_items` → `products` → `category translation`. Three joins, a filter, a date truncation, a group-by and a sort. This is the bread and butter of data engineering work.

> Note: the translation CSV starts with a hidden UTF-8 BOM. We pass explicit column names so both libraries join cleanly. Exercise 4 digs into it.

In [ ]:
TRANS_COLS = ["product_category_name", "product_category_name_english"]

def pandas_query():
    orders = pd.read_csv(DATA / "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
    items = pd.read_csv(DATA / "olist_order_items_dataset.csv")
    products = pd.read_csv(DATA / "olist_products_dataset.csv")
    trans = pd.read_csv(DATA / "product_category_name_translation.csv", names=TRANS_COLS, header=0)

    delivered = orders.loc[orders["order_status"] == "delivered", ["order_id", "order_purchase_timestamp"]]
    df = (
        items.merge(delivered, on="order_id")
        .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
        .merge(trans, on="product_category_name", how="left")
    )
    df["month"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
    return (
        df.groupby(["product_category_name_english", "month"], as_index=False, dropna=False)["price"]
        .sum()
        .rename(columns={"price": "revenue"})
        .sort_values(["product_category_name_english", "month"], na_position="first")
    )

with timer("pandas: read + query") as q_pd:
    out_pd = pandas_query()
out_pd.head()

In [ ]:
def polars_query():
    orders = pl.read_csv(DATA / "olist_orders_dataset.csv", try_parse_dates=True)
    items = pl.read_csv(DATA / "olist_order_items_dataset.csv")
    products = pl.read_csv(DATA / "olist_products_dataset.csv")
    trans = pl.read_csv(DATA / "product_category_name_translation.csv", new_columns=TRANS_COLS)

    delivered = orders.filter(pl.col("order_status") == "delivered").select("order_id", "order_purchase_timestamp")
    return (
        items.join(delivered, on="order_id")
        .join(products.select("product_id", "product_category_name"), on="product_id", how="left")
        .join(trans, on="product_category_name", how="left")
        .group_by(
            "product_category_name_english",
            pl.col("order_purchase_timestamp").dt.truncate("1mo").alias("month"),
        )
        .agg(revenue=pl.col("price").sum())
        .sort("product_category_name_english", "month", nulls_last=False)
    )

with timer("polars: read + query") as q_pl:
    out_pl = polars_query()
print(f"speed-up: {q_pd.seconds / q_pl.seconds:.1f}x")
out_pl.head()

In [ ]:
# Trust, then verify: same groups, same total revenue
print("groups  pandas:", len(out_pd), "| polars:", out_pl.height)
print("revenue pandas:", round(out_pd["revenue"].sum(), 2), "| polars:", round(out_pl["revenue"].sum(), 2))

## Experiment 3 — Same query, lazy (a teaser for Day 7)

Change `read_csv` → `scan_csv` and add `.collect()` at the end. Nothing else changes, but Polars now sees the **whole query** before running it and can skip columns and rows it never needs.

In [ ]:
def polars_lazy_query():
    orders = pl.scan_csv(DATA / "olist_orders_dataset.csv", try_parse_dates=True)
    items = pl.scan_csv(DATA / "olist_order_items_dataset.csv")
    products = pl.scan_csv(DATA / "olist_products_dataset.csv")
    trans = pl.scan_csv(DATA / "product_category_name_translation.csv", new_columns=TRANS_COLS)

    delivered = orders.filter(pl.col("order_status") == "delivered").select("order_id", "order_purchase_timestamp")
    return (
        items.join(delivered, on="order_id")
        .join(products.select("product_id", "product_category_name"), on="product_id", how="left")
        .join(trans, on="product_category_name", how="left")
        .group_by(
            "product_category_name_english",
            pl.col("order_purchase_timestamp").dt.truncate("1mo").alias("month"),
        )
        .agg(revenue=pl.col("price").sum())
        .sort("product_category_name_english", "month", nulls_last=False)
    )

with timer("polars lazy: scan + query") as q_lazy:
    out_lazy = polars_lazy_query().collect()
print(f"speed-up vs pandas: {q_pd.seconds / q_lazy.seconds:.1f}x")

In [ ]:
# What did the optimizer do? Look for the column lists under each CSV SCAN (projection pushdown)
print(polars_lazy_query().explain())

## Why Polars is faster: the 4 design choices

| # | Design choice | What it means in practice |
|---|---|---|
| 1 | **Apache Arrow columnar memory** | Strings are stored in contiguous buffers, not as millions of Python objects. That explains the memory gap in Experiment 1. |
| 2 | **Written in Rust and multi-threaded by default** | Reading, joins and group-bys use all your cores. pandas runs mostly on one. |
| 3 | **A query optimizer (lazy API)** | Polars plans the whole query and reads only the columns and rows it needs (Experiment 3). |
| 4 | **Strict types, no index** | No hidden index alignment and no object dtype guessing. What you see in the schema is what you get. |

The fair caveat for the post: on tiny data (a few thousand rows) both are instant and pandas' ecosystem still wins. Polars pays off as data and query complexity grow.

## Exercises

Do these in the cells below. Write down the numbers: they go into the post.

1. **Memory detective.** In the memory table, which table has the biggest pandas/Polars ratio? Look at its dtypes in pandas (`pd_frames[...].dtypes`) and Polars (`pl_frames[...].schema`) and explain why.
2. **Cores at work.** Open Task Manager → Performance → CPU, then re-run Experiment 2 for pandas and Polars. Describe what the CPU graph does for each. Then try `POLARS_MAX_THREADS=1` (set it in the environment **before** importing polars, so restart the kernel) and re-time Polars. How much of the speed-up came from threads?
3. **Read the plan.** In the `explain()` output, which columns does Polars read from `olist_orders_dataset.csv`? How many columns does that file have in total?
4. **The BOM gotcha.** Load `product_category_name_translation.csv` with plain `pd.read_csv` and `pl.read_csv` (no column names given) and print `.columns` for each, using `repr()` on the first name. Did either library keep the hidden `\ufeff` character? What would happen to the join if it did?
5. **Bonus — spot the typos.** Check the column names of `olist_products_dataset.csv`. Real data is messy: the post can use this as a hook for Day 15 (renaming columns).

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4

In [ ]:
# Exercise 5

## Notes for the post

- Load: pandas ___ s vs Polars ___ s (___x)
- Memory: pandas ___ MB vs Polars ___ MB (biggest gap: ___)
- Query: pandas ___ s, Polars eager ___ s, Polars lazy ___ s
- One surprise:
- Machine: CPU ___, cores ___, RAM ___